In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from scipy.stats import pearsonr
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability


In [2]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names =  ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min'] # ignore short sessions (10, 24,25)

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-02-19 14:22:17,051|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-19 14:22:17,419|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-02-19 14:22:17,681|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-19 14:22:17,897|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-19 14:22:17,943|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min.hdf5 excluded
2026-02-19 14:22:18,237|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 30 sessions.
2026-02-19 14:22:18,238|DEBUG|43827|sessions_from_nas_parsing|sessionl

In [3]:
fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)

2026-02-19 14:22:18,256|DEBUG|43827|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-19 14:22:18,534|DEBUG|43827|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-19 14:22:18,535|DEBUG|43827|analytics|get_analytics
	Processing FiringRateTrackwiseHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-19 14:22:18,581|INFO|43827|analytics|get_analytics
	Analytic `FiringRateTrackwiseHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-19 14:22:18,582|DEBUG|43827|analytics|get_analytics
	Processing FiringRateTrackwiseHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrack

In [ ]:
# type classification (cue / reward-approach / reward)
# observed mean activity in feature-specific windows vs. 1,000 circular shuffles (99th percentile threshold).

N_SHUFFLES = 1000
THRESHOLD_PERCENTILE = 99
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

required_cols = {"from_position_bin", "trial_id", "cue", "choice_R1", "choice_R2"}
missing_cols = sorted(required_cols - set(fr_track.columns))
if missing_cols:
    raise KeyError(f"fr_track is missing required columns: {missing_cols}")

unit_cols = [c for c in fr_track.columns if c.startswith("Unit")]
if not unit_cols:
    raise ValueError("No unit columns found in fr_track (expected columns starting with 'Unit').")


def _coerce_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    txt = series.astype(str).str.strip().str.lower()
    mapped = txt.map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "1.0": True,
            "0.0": False,
            "t": True,
            "f": False,
        }
    )
    if mapped.notna().sum() > 0:
        return mapped.fillna(False)

    num = pd.to_numeric(series, errors="coerce")
    if num.notna().sum() > 0:
        return num.fillna(0).ne(0)

    return series.fillna(False).astype(bool)


def _safe_mean(x: np.ndarray, mask: np.ndarray) -> np.ndarray:
    if mask.any():
        return np.nanmean(x[mask, :], axis=0)
    return np.full(x.shape[1], np.nan, dtype=float)


records = []
session_ids = fr_track.index.get_level_values("session_id").unique()

for session_id in session_ids:
    sess = fr_track.xs(session_id, level="session_id").copy()
    sess = sess.sort_values(["trial_id", "from_position_bin"]).reset_index(drop=True)

    pos = pd.to_numeric(sess["from_position_bin"], errors="coerce")
    cue = pd.to_numeric(sess["cue"], errors="coerce")
    trial_ok = sess["trial_id"].notna() & pos.notna()

    if not trial_ok.any():
        continue

    sess = sess.loc[trial_ok].reset_index(drop=True)
    pos = pd.to_numeric(sess["from_position_bin"], errors="coerce")
    cue = pd.to_numeric(sess["cue"], errors="coerce")
    choice_r1 = _coerce_bool(sess["choice_R1"])
    choice_r2 = _coerce_bool(sess["choice_R2"])

    # Reward exists only when cue matches the rewarded choice.
    rewarded_trial = ((cue == 1) & choice_r1) | ((cue == 2) & choice_r2)

    masks = {
        # Cue zone
        "cue": (pos >= -80) & (pos <= 25),
        # Pre-reward zone before expected reward location (rewarded trials only)
        "reward_approach": (
            ((cue == 1) & rewarded_trial & (pos > 25) & (pos < 50))
            | ((cue == 2) & rewarded_trial & (pos > 110) & (pos < 170))
        ),
        # Reward zone at actually rewarded target (rewarded trials only)
        "reward": (
            ((cue == 1) & rewarded_trial & (pos >= 50) & (pos <= 110))
            | ((cue == 2) & rewarded_trial & (pos >= 170) & (pos <= 230))
        ),
    }

    x = sess[unit_cols].to_numpy(dtype=float, copy=True)
    n_rows, n_units = x.shape

    trial_ids = sess["trial_id"].to_numpy()
    unique_trials = pd.unique(trial_ids)
    trial_row_idx = [np.flatnonzero(trial_ids == tid) for tid in unique_trials]

    observed = {k: _safe_mean(x, m.to_numpy()) for k, m in masks.items()}

    shuffled = {
        "cue": np.full((N_SHUFFLES, n_units), np.nan, dtype=float),
        "reward_approach": np.full((N_SHUFFLES, n_units), np.nan, dtype=float),
        "reward": np.full((N_SHUFFLES, n_units), np.nan, dtype=float),
    }

    for s in range(N_SHUFFLES):
        x_shuf = np.empty_like(x)
        for rows in trial_row_idx:
            if len(rows) <= 1:
                x_shuf[rows, :] = x[rows, :]
            else:
                shift = int(rng.integers(0, len(rows)))
                x_shuf[rows, :] = np.roll(x[rows, :], shift=shift, axis=0)

        for key, mask in masks.items():
            mask_np = mask.to_numpy()
            if mask_np.any():
                shuffled[key][s, :] = np.nanmean(x_shuf[mask_np, :], axis=0)

    for i, unit in enumerate(unit_cols):
        rec = {
            "session_id": session_id,
            "unit": unit,
            "n_rows_session": int(n_rows),
            "n_rows_cue": int(masks["cue"].sum()),
            "n_rows_reward_approach": int(masks["reward_approach"].sum()),
            "n_rows_reward": int(masks["reward"].sum()),
        }

        for key in ["cue", "reward_approach", "reward"]:
            obs = observed[key][i]
            sh = shuffled[key][:, i]
            valid = np.isfinite(sh)

            if valid.any() and np.isfinite(obs):
                thr = float(np.percentile(sh[valid], THRESHOLD_PERCENTILE))
                pval = float((np.sum(sh[valid] >= obs) + 1) / (valid.sum() + 1))
            else:
                thr = np.nan
                pval = np.nan

            is_cell = bool(np.isfinite(obs) and np.isfinite(thr) and (obs > thr))
            rec[f"obs_{key}"] = float(obs) if np.isfinite(obs) else np.nan
            rec[f"thr99_{key}"] = thr
            rec[f"p_{key}"] = pval
            rec[f"is_{key}_cell"] = is_cell

        labels = []
        if rec["is_cue_cell"]:
            labels.append("cue")
        if rec["is_reward_approach_cell"]:
            labels.append("reward_approach")
        if rec["is_reward_cell"]:
            labels.append("reward")
        rec["assigned_label"] = "+".join(labels) if labels else "none"

        records.append(rec)


classification_trackwise = (
    pd.DataFrame(records)
    .sort_values(["session_id", "unit"])
    .reset_index(drop=True)
)

summary_counts = (
    classification_trackwise[
        ["is_cue_cell", "is_reward_approach_cell", "is_reward_cell"]
    ]
    .sum()
    .rename("n_units")
    .to_frame()
)

print(f"Computed classification for {len(classification_trackwise)} unit-session entries.")
display(summary_counts)
display(classification_trackwise.head())

Computed classification for 1925 unit-session entries.


,n_units
is_cue_cell,198
is_reward_approach_cell,144
is_reward_cell,394


,session_id,unit,n_rows_session,n_rows_cue,n_rows_reward_approach,n_rows_reward,obs_cue,thr99_cue,p_cue,is_cue_cell,obs_reward_approach,thr99_reward_approach,p_reward_approach,is_reward_approach_cell,obs_reward,thr99_reward,p_reward,is_reward_cell,assigned_label
0,2024-11-14_16-40,Unit0001,44005,10812,465,915,0.637407,1.441457,1.000000,False,0.528674,1.970407,0.979021,False,1.992676,1.954221,0.007992,True,reward
1,2024-11-14_16-40,Unit0002,44005,10812,465,915,0.242450,0.600107,1.000000,False,0.806452,1.026567,0.064935,False,0.634845,0.894084,0.176823,False,none
2,2024-11-14_16-40,Unit0003,44005,10812,465,915,0.197432,0.299623,0.810190,False,0.456989,0.675057,0.108891,False,0.219612,0.489939,0.457542,False,none
3,2024-11-14_16-40,Unit0004,44005,10812,465,915,0.005781,0.032438,0.885115,False,0.134409,0.152599,0.048951,False,0.043716,0.122951,0.205794,False,none
4,2024-11-14_16-40,Unit0005,44005,10812,465,915,0.215636,0.580975,1.000000,False,0.501792,0.970298,0.300699,False,0.642027,0.875754,0.132867,False,none


In [5]:
# --- Plots: encoding certainty and session dynamics

if "classification_trackwise" not in globals():
    raise NameError("Run the classification cell first so classification_trackwise exists.")

dfc = classification_trackwise.copy()

features = ["cue", "reward_approach", "reward"]
obs_cols = [f"obs_{f}" for f in features]
p_cols = [f"p_{f}" for f in features]
required = {"session_id", "unit", *obs_cols, *p_cols}
missing = sorted(required - set(dfc.columns))
if missing:
    raise KeyError(f"classification_trackwise is missing required columns: {missing}")

obs = dfc[obs_cols].to_numpy(dtype=float)
pvals = dfc[p_cols].to_numpy(dtype=float)

obs_filled = np.where(np.isfinite(obs), obs, -np.inf)
pref_idx = np.argmax(obs_filled, axis=1)
pref_obs = obs[np.arange(len(dfc)), pref_idx]

runner = obs_filled.copy()
runner[np.arange(len(dfc)), pref_idx] = -np.inf
runner_obs = np.max(runner, axis=1)
runner_obs = np.where(np.isfinite(runner_obs), runner_obs, np.nan)

# Selectivity against the next-best zone; close to 1 means strongly one-zone dominant.
selectivity_index = (pref_obs - runner_obs) / (
    np.abs(pref_obs) + np.abs(runner_obs) + 1e-12
)

pref_p = pvals[np.arange(len(dfc)), pref_idx]
shuffle_confidence = 1 - pref_p
is_significant_pref = pref_p < 0.01
preferred_feature = np.array(features, dtype=object)[pref_idx]

certainty_df = dfc[["session_id", "unit"]].copy()
certainty_df["preferred_feature"] = preferred_feature
certainty_df["pref_obs"] = pref_obs
certainty_df["runner_obs"] = runner_obs
certainty_df["selectivity_index"] = selectivity_index
certainty_df["pref_p"] = pref_p
certainty_df["shuffle_confidence"] = shuffle_confidence
certainty_df["certainty_score"] = certainty_df["selectivity_index"] * certainty_df["shuffle_confidence"]
certainty_df["is_significant_pref"] = is_significant_pref

# Useful check for your earlier question: rows are unit-session entries.
print(
    f"unit-session rows: {len(certainty_df)} | unique units: {certainty_df['unit'].nunique()} | "
    f"unique sessions: {certainty_df['session_id'].nunique()}"
)

# Plot 1: certainty of cell kind
fig_certainty = px.scatter(
    certainty_df,
    x="selectivity_index",
    y="shuffle_confidence",
    color="preferred_feature",
    symbol="is_significant_pref",
    opacity=0.7,
    hover_data={
        "session_id": True,
        "unit": True,
        "pref_obs": ':.4f',
        "runner_obs": ':.4f',
        "pref_p": ':.4g',
        "certainty_score": ':.4f',
    },
    title="Encoding certainty per unit-session (selectivity vs shuffle confidence)",
    labels={
        "selectivity_index": "Selectivity vs next-best zone",
        "shuffle_confidence": "Shuffle confidence (1 - p)",
        "preferred_feature": "Preferred encoding",
        "is_significant_pref": "p < 0.01",
    },
)
fig_certainty.update_layout(template="plotly_white")
fig_certainty.show()

# Plot 2: session-wise change in encoding prevalence
flag_map = {
    "is_cue_cell": "cue",
    "is_reward_approach_cell": "reward_approach",
    "is_reward_cell": "reward",
}
missing_flags = sorted(set(flag_map) - set(dfc.columns))
if missing_flags:
    raise KeyError(f"classification_trackwise is missing class flags: {missing_flags}")

session_frac = (
    dfc.groupby("session_id", as_index=False)[list(flag_map.keys())]
    .mean()
    .rename(columns=flag_map)
)

session_long = session_frac.melt(
    id_vars="session_id",
    var_name="cell_type",
    value_name="fraction_units",
)
session_long["percent_units"] = 100.0 * session_long["fraction_units"]

session_long["session_dt"] = pd.to_datetime(session_long["session_id"], errors="coerce")
if session_long["session_dt"].notna().any():
    session_long = session_long.sort_values(["session_dt", "cell_type"])
else:
    session_long = session_long.sort_values(["session_id", "cell_type"])

fig_dynamics = px.line(
    session_long,
    x="session_id",
    y="percent_units",
    color="cell_type",
    markers=True,
    title="Encoding prevalence across sessions (% classified units)",
    labels={
        "session_id": "Session",
        "percent_units": "% of units",
        "cell_type": "Encoding type",
    },
)
fig_dynamics.update_layout(template="plotly_white")
fig_dynamics.update_yaxes(range=[0, 100])
fig_dynamics.show()



unit-session rows: 1925 | unique units: 77 | unique sessions: 25


/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_43827/602034658.py:104: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [6]:
meta_data = {}
meta_data['SpikeClusterMetadata'] = analytics.get_analytics('SpikeClusterMetadata', mode='set',
                                                      #  columns = cols,
                                                       paradigm_ids=paradigm,
                                                       animal_ids=animal_ids,
                                                       excl_session_names=excl_session_names,
                                                       session_ids=session_ids)


meta_data['SpikeClusterMetadata']

2026-02-19 14:29:33,655|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-19 14:29:34,741|DEBUG|43827|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 25 sessions.
2026-02-19 14:29:34,743|DEBUG|43827|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: [1100], animal_ids: [6], session_ids: Index(['2024-11-14_16-40', '2024-11-15_15-48', '2024-11-20_17-46',
       '2024-11-21_17-22', '2024-11-25_16-25', '2024-11-26_16-39',
       '2024-11-28_17-41', '2024-12-02_16-09', '2024-12-03_16-23',
       '2024-12-04_18-06', '2024-12-06_16-49', '2024-12-09_17-45',
       '2024-12-10_17-20', '2024-12-12_16-13', '2024-12-13_17-10',
       '2025-01-14_18-08', '2025-01-15_17-18', '2025-01-16_17-47',
       '2025-01-17_16-55', '2025-01-23_16-48', '2025-01-24_12-24',
       '2025-01-24_19-37', '2025-01-25_21-29', '2025-01-26_21-48',
       '2025-01-27_13-39'],
      dtype='

cluster_id cluster_type  \
paradigm_id animal_id session_id       entry_id                            
1100        6         2024-11-14_16-40 0                  1       single   
                                       1                  2        3-ISI   
                                       2                  3       single   
                                       3                  4        5-ISI   
                                       4                  5       single   
...                                                     ...          ...   
                      2025-01-27_13-39 72                73       single   
                                       73                74       single   
                                       74                75       single   
                                       75                76       single   
                                       76                77       single   

                                                 unit_count  cluster_channel  \
paradigm_id animal_id session_id       entry_id                                
1100        6         2024-11-14_16-40 0            44619.0            295.0   
                                       1           208600.0            299.0   
                                       2            11050.0            297.0   
                                       3            18939.0            309.0   
                                       4            23034.0            311.0   
...                                                     ...              ...   
                      2025-01-27_13-39 72           28147.0            150.0   
                                       73          585584.0            151.0   
                                       74          370562.0            154.0   
                                       75          272688.0            156.0   
                                       76          200568.0            158.0   

                                                 cluster_id_ssbatch  unit_snr  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0                          1  6.475497   
                                       1                          2  4.558548   
                                       2                          3  4.637581   
                                       3                          4  5.504494   
                                       4                          5  5.848962   
...                                                             ...       ...   
                      2025-01-27_13-39 72                        56  6.038227   
                                       73                        57  7.484550   
                                       74                        58  6.215347   
                                       75                        59  6.829296   
                                       76                        60  6.174706   

                                                  unit_Vpp  unit_isi_ratio  \
paradigm_id animal_id session_id       entry_id                              
1100        6         2024-11-14_16-40 0         61.824997        0.173228   
                                       1         53.466000        0.722572   
                                       2         55.282997        0.517730   
                                       3         53.279999        0.492537   
                                       4         49.151997        0.198254   
...                                                    ...             ...   
                      2025-01-27_13-39 72        77.212997        0.156766   
                                       73        81.588997        0.079114   
                                       74        65.533997        0.094184   
                                       75        68.143005        0.070829   
                             

In [11]:
# Region-level encoding and stability analysis
# Uses classification_trackwise + meta_data['SpikeClusterMetadata'] to localize encoding patterns.

MIN_SIG_SESSIONS_STABLE = 3

dfc = classification_trackwise.copy()
meta_spike = meta_data["SpikeClusterMetadata"].copy()

# Rebuild certainty table if the earlier plotting cell has not been run.
if "certainty_df" not in globals():
    features = ["cue", "reward_approach", "reward"]
    obs_cols = [f"obs_{f}" for f in features]
    p_cols = [f"p_{f}" for f in features]

    obs = dfc[obs_cols].to_numpy(dtype=float)
    pvals = dfc[p_cols].to_numpy(dtype=float)
    obs_filled = np.where(np.isfinite(obs), obs, -np.inf)

    pref_idx = np.argmax(obs_filled, axis=1)
    pref_obs = obs[np.arange(len(dfc)), pref_idx]

    runner = obs_filled.copy()
    runner[np.arange(len(dfc)), pref_idx] = -np.inf
    runner_obs = np.max(runner, axis=1)
    runner_obs = np.where(np.isfinite(runner_obs), runner_obs, np.nan)

    selectivity_index = (pref_obs - runner_obs) / (
        np.abs(pref_obs) + np.abs(runner_obs) + 1e-12
    )

    pref_p = pvals[np.arange(len(dfc)), pref_idx]
    shuffle_confidence = 1 - pref_p
    is_significant_pref = pref_p < 0.01
    preferred_feature = np.array(features, dtype=object)[pref_idx]

    certainty_df = dfc[["session_id", "unit"]].copy()
    certainty_df["preferred_feature"] = preferred_feature
    certainty_df["pref_p"] = pref_p
    certainty_df["is_significant_pref"] = is_significant_pref
    certainty_df["selectivity_index"] = selectivity_index
    certainty_df["shuffle_confidence"] = shuffle_confidence
    certainty_df["certainty_score"] = selectivity_index * shuffle_confidence


# ------------------------
# Build metadata join keys
# ------------------------
mdf = meta_spike.reset_index().copy()

if "session_id" not in mdf.columns:
    raise KeyError("SpikeClusterMetadata must contain session_id (as column or index level).")

area_candidates = ["fine_brain_area", "brain_area", "region", "area"]
area_col = next((c for c in area_candidates if c in mdf.columns), None)
if area_col is None:
    raise KeyError(
        "Could not find brain region column. Expected one of: "
        f"{area_candidates} in SpikeClusterMetadata columns."
    )

mdf["brain_region"] = (
    mdf[area_col]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    .fillna("Unknown")
)


def _normalize_unit_from_series(series: pd.Series) -> pd.Series:
    txt = series.astype(str).str.strip()

    # Parse labels like Unit0007 / unit 7
    from_label = txt.str.extract(r"(?i)unit\s*0*(\d+)", expand=False)
    from_label_num = pd.to_numeric(from_label, errors="coerce")

    # Parse pure numeric ids
    from_num = pd.to_numeric(series, errors="coerce")

    unit_num = from_label_num.combine_first(from_num)
    out = unit_num.map(lambda v: f"Unit{int(v):04d}" if pd.notna(v) else np.nan)
    return out


unit_col = None
for candidate in ["unit", "unit_id", "unit_name", "cluster_id", "entry_id"]:
    if candidate in mdf.columns:
        temp = _normalize_unit_from_series(mdf[candidate])
        if temp.notna().any():
            unit_col = candidate
            mdf["unit"] = temp
            break

if "unit" not in mdf.columns or mdf["unit"].notna().sum() == 0:
    raise KeyError(
        "Could not build unit labels from SpikeClusterMetadata. "
        "Expected one of unit/unit_id/unit_name/cluster_id/entry_id."
    )

# entry_id is often 0-indexed; adjust only if everything parsed from entry_id looked like Unit0000...
if unit_col == "entry_id":
    mdf_entry = _normalize_unit_from_series(mdf["entry_id"] + 1)
    n_match_default = mdf["unit"].isin(dfc["unit"].unique()).sum()
    n_match_shifted = mdf_entry.isin(dfc["unit"].unique()).sum()
    if n_match_shifted > n_match_default:
        mdf["unit"] = mdf_entry

meta_map = mdf[["session_id", "unit", "brain_region"]].dropna(subset=["session_id", "unit"]).copy()
meta_map = (
    meta_map.groupby(["session_id", "unit"], as_index=False)["brain_region"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0])
)

# -------------------------
# Merge classification + region
# -------------------------
cert_small = certainty_df[
    [
        "session_id",
        "unit",
        "preferred_feature",
        "is_significant_pref",
        "pref_p",
        "certainty_score",
    ]
].copy()

row_df = (
    dfc.merge(cert_small, on=["session_id", "unit"], how="left")
    .merge(meta_map, on=["session_id", "unit"], how="left")
)
row_df["brain_region"] = row_df["brain_region"].fillna("Unknown")

row_df["encoding_state"] = np.where(
    row_df["is_significant_pref"].fillna(False),
    row_df["preferred_feature"],
    "none",
)

# Session sort order
session_order = pd.DataFrame({"session_id": row_df["session_id"].drop_duplicates()})
session_order["session_dt"] = pd.to_datetime(
    session_order["session_id"], format="%Y-%m-%d_%H-%M", errors="coerce"
)
if session_order["session_dt"].notna().all():
    session_order = session_order.sort_values("session_dt")
else:
    session_order = session_order.sort_values("session_id")
session_order["session_order"] = np.arange(len(session_order))
row_df = row_df.merge(session_order[["session_id", "session_order"]], on="session_id", how="left")


def _dominant_encoding(g: pd.DataFrame) -> str:
    sig = g[g["encoding_state"] != "none"]
    if sig.empty:
        return "none"

    counts = sig["encoding_state"].value_counts()
    top = counts[counts == counts.max()].index.tolist()
    if len(top) == 1:
        return top[0]

    # Tie-break with higher mean certainty score.
    tie = (
        sig[sig["encoding_state"].isin(top)]
        .groupby("encoding_state", as_index=False)["certainty_score"]
        .mean()
        .sort_values("certainty_score", ascending=False)
    )
    return tie.iloc[0]["encoding_state"]


def _unit_summary(g: pd.DataFrame) -> pd.Series:
    sig = g[g["encoding_state"] != "none"]
    n_sig = int(len(sig))
    n_unique = int(sig["encoding_state"].nunique()) if n_sig > 0 else 0

    dom = _dominant_encoding(g)
    if n_sig == 0:
        stability = "unclassified"
    elif n_unique == 1 and n_sig >= MIN_SIG_SESSIONS_STABLE:
        stability = "stable"
    elif n_unique == 1 and n_sig < MIN_SIG_SESSIONS_STABLE:
        stability = "low_support"
    else:
        stability = "changing"

    region_mode = g["brain_region"].mode()
    region = region_mode.iat[0] if not region_mode.empty else g["brain_region"].iloc[0]

    return pd.Series(
        {
            "brain_region": region,
            "dominant_encoding": dom,
            "stability_class": stability,
            "n_sessions": int(g["session_id"].nunique()),
            "n_sig_sessions": n_sig,
            "n_unique_sig_encodings": n_unique,
        }
    )


unit_profile = row_df.groupby("unit", as_index=False).apply(_unit_summary).reset_index(drop=True)

print(
    f"Units total: {len(unit_profile)} | "
    f"stable: {(unit_profile['stability_class'] == 'stable').sum()} | "
    f"changing: {(unit_profile['stability_class'] == 'changing').sum()} | "
    f"unclassified: {(unit_profile['stability_class'] == 'unclassified').sum()}"
)

# ----------------------
# Plot 1: region x encoding (unit-level dominant encoding)
# ----------------------
enc_order = ["cue", "reward_approach", "reward", "none"]
region_order = (
    unit_profile["brain_region"].value_counts().sort_values(ascending=False).index.tolist()
)

region_encoding_counts = (
    unit_profile.groupby(["brain_region", "dominant_encoding"], as_index=False)
    .size()
    .rename(columns={"size": "n_units"})
)

fig_region_encoding = px.bar(
    region_encoding_counts,
    x="brain_region",
    y="n_units",
    color="dominant_encoding",
    barmode="stack",
    category_orders={"dominant_encoding": enc_order, "brain_region": region_order},
    title="Neuron counts by brain region and dominant encoding",
    labels={
        "brain_region": "Brain region",
        "n_units": "Number of neurons",
        "dominant_encoding": "Dominant encoding",
    },
)
fig_region_encoding.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_region_encoding.show()

# ----------------------
# Plot 2: region x stability (stable/changing)
# ----------------------
stab_order = ["stable", "changing", "low_support", "unclassified"]
region_stability_counts = (
    unit_profile.groupby(["brain_region", "stability_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_units"})
)

fig_region_stability = px.bar(
    region_stability_counts,
    x="brain_region",
    y="n_units",
    color="stability_class",
    barmode="stack",
    category_orders={"stability_class": stab_order, "brain_region": region_order},
    title="Stable vs changing neurons by brain region",
    labels={
        "brain_region": "Brain region",
        "n_units": "Number of neurons",
        "stability_class": "Encoding stability",
    },
)
fig_region_stability.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_region_stability.show()

# ----------------------
# Plot 3 (refined): per-neuron encoding timeline across sessions
# ----------------------
timeline = row_df[["unit", "session_id", "session_order", "encoding_state", "brain_region"]].drop_duplicates(
    subset=["unit", "session_id"]
)
ordered_sessions = (
    session_order.sort_values("session_order")["session_id"].tolist()
)

stab_rank = {"changing": 0, "stable": 1, "low_support": 2, "unclassified": 3}
unit_profile_plot = unit_profile.copy()
unit_profile_plot["_stab_rank"] = unit_profile_plot["stability_class"].map(stab_rank).fillna(99)
unit_profile_plot = unit_profile_plot.sort_values(["_stab_rank", "brain_region", "unit"])
unit_order = unit_profile_plot["unit"].tolist()

state_matrix = timeline.pivot(index="unit", columns="session_id", values="encoding_state")
state_matrix = state_matrix.reindex(index=unit_order, columns=ordered_sessions)

state_to_int = {"none": 0, "cue": 1, "reward_approach": 2, "reward": 3}
z = state_matrix.replace(state_to_int).fillna(0).to_numpy()

label_map = unit_profile_plot.set_index("unit").apply(
    lambda r: f"{r.name} | {r['brain_region']} | {r['stability_class']}", axis=1
)
y_labels = [label_map.get(u, u) for u in state_matrix.index]

colorscale = [
    [0.00, "#d9d9d9"], [0.24, "#d9d9d9"],
    [0.25, "#1f77b4"], [0.49, "#1f77b4"],
    [0.50, "#ff7f0e"], [0.74, "#ff7f0e"],
    [0.75, "#2ca02c"], [1.00, "#2ca02c"],
]

fig_timeline = go.Figure(
    data=go.Heatmap(
        z=z,
        x=state_matrix.columns,
        y=y_labels,
        customdata=state_matrix.to_numpy(),
        colorscale=colorscale,
        zmin=0,
        zmax=3,
        colorbar=dict(
            title="Encoding",
            tickvals=[0, 1, 2, 3],
            ticktext=["none", "cue", "reward_approach", "reward"],
        ),
        hovertemplate="unit=%{y}<br>session=%{x}<br>encoding=%{customdata}<extra></extra>",
    )
)
fig_timeline.update_layout(
    template="plotly_white",
    title="Encoding timeline per neuron (sorted by changing/stable and brain region)",
    xaxis_title="Session",
    yaxis_title="Neuron | Region | Stability",
    height=max(500, 14 * len(y_labels)),
)
fig_timeline.update_xaxes(tickangle=-35)
fig_timeline.show()

# Optional quick tables for direct lookup
stable_units = unit_profile[unit_profile["stability_class"] == "stable"].sort_values(["brain_region", "unit"])
changing_units = unit_profile[unit_profile["stability_class"] == "changing"].sort_values(["brain_region", "unit"])

display(stable_units.head(20))
display(changing_units.head(20))



/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_43827/3118060616.py:80: FutureWarning:

The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.



Units total: 77 | stable: 11 | changing: 57 | unclassified: 2


/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_43827/3118060616.py:203: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_43827/3118060616.py:290: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,unit,brain_region,dominant_encoding,stability_class,n_sessions,n_sig_sessions,n_unique_sig_encodings
20,Unit0021,ACC,reward_approach,stable,25,8,1
0,Unit0001,CA1,reward,stable,25,9,1
4,Unit0005,DG,reward,stable,25,4,1
17,Unit0018,DG,reward,stable,25,10,1
66,Unit0067,InfrL,reward,stable,25,7,1
67,Unit0068,InfrL,reward,stable,25,14,1
70,Unit0071,InfrL,cue,stable,25,11,1
74,Unit0075,InfrL,reward,stable,25,5,1
28,Unit0029,PrL,cue,stable,25,6,1
29,Unit0030,PrL,reward,stable,25,4,1


,unit,brain_region,dominant_encoding,stability_class,n_sessions,n_sig_sessions,n_unique_sig_encodings
21,Unit0022,ACC,reward_approach,changing,25,14,3
22,Unit0023,ACC,reward,changing,25,13,3
23,Unit0024,ACC,reward_approach,changing,25,18,3
24,Unit0025,ACC,reward_approach,changing,25,14,2
25,Unit0026,ACC,reward_approach,changing,25,9,2
26,Unit0027,ACC,reward,changing,25,16,2
27,Unit0028,ACC,reward_approach,changing,25,3,3
1,Unit0002,CA1,reward_approach,changing,25,12,2
2,Unit0003,CA1,reward,changing,25,3,3
3,Unit0004,DG,reward,changing,25,4,2


In [ ]:
# Overall neuron classification across all sessions (pooled longitudinally)
# One classification per tracked neuron (UnitXXXX) by pooling all sessions,
# while keeping the same circular shuffle logic (within each trial, per session).

N_SHUFFLES_OVERALL = 1000
THRESHOLD_PERCENTILE = 99
RNG_SEED_OVERALL = 123
rng_overall = np.random.default_rng(RNG_SEED_OVERALL)

unit_cols = [c for c in fr_track.columns if c.startswith("Unit")]

def _coerce_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    txt = series.astype(str).str.strip().str.lower()
    mapped = txt.map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "1.0": True,
            "0.0": False,
            "t": True,
            "f": False,
        }
    )
    if mapped.notna().sum() > 0:
        return mapped.fillna(False)

    num = pd.to_numeric(series, errors="coerce")
    if num.notna().sum() > 0:
        return num.fillna(0).ne(0)

    return series.fillna(False).astype(bool)


def _sum_and_count(arr: np.ndarray):
    return np.nansum(arr, axis=0), np.sum(np.isfinite(arr), axis=0)


feature_keys = ["cue", "reward_approach", "reward"]
n_units = len(unit_cols)

# Precompute session payloads once
session_payloads = []
for session_id in fr_track.index.get_level_values("session_id").unique():
    sess = fr_track.xs(session_id, level="session_id").copy()
    sess = sess.sort_values(["trial_id", "from_position_bin"]).reset_index(drop=True)

    pos = pd.to_numeric(sess["from_position_bin"], errors="coerce")
    cue = pd.to_numeric(sess["cue"], errors="coerce")
    trial_ok = sess["trial_id"].notna() & pos.notna()

    # if not trial_ok.any():
    #     continue

    sess = sess.loc[trial_ok].reset_index(drop=True)
    pos = pd.to_numeric(sess["from_position_bin"], errors="coerce")
    cue = pd.to_numeric(sess["cue"], errors="coerce")
    choice_r1 = _coerce_bool(sess["choice_R1"])
    choice_r2 = _coerce_bool(sess["choice_R2"])

    rewarded_trial = ((cue == 1) & choice_r1) | ((cue == 2) & choice_r2)

    masks = {
        "cue": ((pos >= -80) & (pos <= 25)).to_numpy(),
        "reward_approach": (
            ((cue == 1) & rewarded_trial & (pos > 25) & (pos < 50))
            | ((cue == 2) & rewarded_trial & (pos > 110) & (pos < 170))
        ).to_numpy(),
        "reward": (
            ((cue == 1) & rewarded_trial & (pos >= 50) & (pos <= 110))
            | ((cue == 2) & rewarded_trial & (pos >= 170) & (pos <= 230))
        ).to_numpy(),
    }

    x = sess[unit_cols].to_numpy(dtype=float, copy=True)

    trial_ids = sess["trial_id"].to_numpy()
    unique_trials = pd.unique(trial_ids)
    trial_row_idx = [np.flatnonzero(trial_ids == tid) for tid in unique_trials]

    session_payloads.append(
        {
            "session_id": session_id,
            "x": x,
            "masks": masks,
            "trial_row_idx": trial_row_idx,
        }
    )

# Observed pooled means
obs_sum = {k: np.zeros(n_units, dtype=float) for k in feature_keys}
obs_cnt = {k: np.zeros(n_units, dtype=int) for k in feature_keys}

for payload in session_payloads:
    x = payload["x"]
    masks = payload["masks"]
    for key in feature_keys:
        vals = x[masks[key], :]
        s, c = _sum_and_count(vals)
        obs_sum[key] += s
        obs_cnt[key] += c

observed = {
    k: np.divide(
        obs_sum[k],
        obs_cnt[k],
        out=np.full(n_units, np.nan, dtype=float),
        where=obs_cnt[k] > 0,
    )
    for k in feature_keys
}

# Shuffle pooled means
shuffled = {k: np.full((N_SHUFFLES_OVERALL, n_units), np.nan, dtype=float) for k in feature_keys}

for s in range(N_SHUFFLES_OVERALL):
    shuf_sum = {k: np.zeros(n_units, dtype=float) for k in feature_keys}
    shuf_cnt = {k: np.zeros(n_units, dtype=int) for k in feature_keys}

    for payload in session_payloads:
        x = payload["x"]
        masks = payload["masks"]
        trial_row_idx = payload["trial_row_idx"]

        x_shuf = np.empty_like(x)
        for rows in trial_row_idx:
            if len(rows) <= 1:
                x_shuf[rows, :] = x[rows, :]
            else:
                shift = int(rng_overall.integers(0, len(rows)))
                x_shuf[rows, :] = np.roll(x[rows, :], shift=shift, axis=0)

        for key in feature_keys:
            vals = x_shuf[masks[key], :]
            s_part, c_part = _sum_and_count(vals)
            shuf_sum[key] += s_part
            shuf_cnt[key] += c_part

    for key in feature_keys:
        shuffled[key][s, :] = np.divide(
            shuf_sum[key],
            shuf_cnt[key],
            out=np.full(n_units, np.nan, dtype=float),
            where=shuf_cnt[key] > 0,
        )

    if (s + 1) % 100 == 0 or (s + 1) == N_SHUFFLES_OVERALL:
        print(f"Overall shuffle {s + 1}/{N_SHUFFLES_OVERALL}")

# Final overall classification per neuron
records = []
for i, unit in enumerate(unit_cols):
    rec = {"unit": unit}
    labels = []

    for key in feature_keys:
        obs = observed[key][i]
        sh = shuffled[key][:, i]
        valid = np.isfinite(sh)

        if valid.any() and np.isfinite(obs):
            thr = float(np.percentile(sh[valid], THRESHOLD_PERCENTILE))
            pval = float((np.sum(sh[valid] >= obs) + 1) / (valid.sum() + 1))
        else:
            thr = np.nan
            pval = np.nan

        is_cell = bool(np.isfinite(obs) and np.isfinite(thr) and (obs > thr))
        rec[f"obs_{key}"] = float(obs) if np.isfinite(obs) else np.nan
        rec[f"thr99_{key}"] = thr
        rec[f"p_{key}"] = pval
        rec[f"is_{key}_cell"] = is_cell
        if is_cell:
            labels.append(key)

    rec["assigned_label"] = "+".join(labels) if labels else "none"
    rec["n_sig_types"] = len(labels)
    records.append(rec)

classification_overall = (
    pd.DataFrame(records)
    .sort_values("unit")
    .reset_index(drop=True)
)

summary_overall = (
    classification_overall[["is_cue_cell", "is_reward_approach_cell", "is_reward_cell"]]
    .sum()
    .rename("n_units")
    .to_frame()
)

print(f"Computed overall classification for {len(classification_overall)} neurons.")
display(summary_overall)
display(classification_overall.head())



Overall shuffle 100/1000
Overall shuffle 200/1000
Overall shuffle 300/1000
Overall shuffle 400/1000
Overall shuffle 500/1000
Overall shuffle 600/1000
Overall shuffle 700/1000
Overall shuffle 800/1000
Overall shuffle 900/1000
Overall shuffle 1000/1000
Computed overall classification for 77 neurons.


,n_units
is_cue_cell,19
is_reward_approach_cell,17
is_reward_cell,35


,unit,obs_cue,thr99_cue,p_cue,is_cue_cell,obs_reward_approach,thr99_reward_approach,p_reward_approach,is_reward_approach_cell,obs_reward,thr99_reward,p_reward,is_reward_cell,assigned_label,n_sig_types
0,Unit0001,0.751841,0.935860,1.000000,False,0.898115,1.009338,0.873127,False,0.707998,0.843396,1.000000,False,none,0
1,Unit0002,2.579212,3.112830,1.000000,False,3.973236,3.275400,0.000999,True,3.095722,3.295882,0.945055,False,reward_approach,1
2,Unit0003,0.165815,0.200625,0.999001,False,0.195684,0.174395,0.000999,True,0.183624,0.197743,0.169830,False,reward_approach,1
3,Unit0004,0.304614,0.328230,0.878122,False,0.455095,0.440642,0.003996,True,0.334987,0.370927,0.579421,False,reward_approach,1
4,Unit0005,0.269434,0.403816,1.000000,False,0.408223,0.463604,0.563437,False,0.479374,0.433379,0.000999,True,reward,1


In [ ]:
# --- Overall classification plots: neuron identity, location, confidence

ov = classification_overall.copy()
features = ["cue", "reward_approach", "reward"]
obs_cols = [f"obs_{f}" for f in features]
thr_cols = [f"thr99_{f}" for f in features]
p_cols = [f"p_{f}" for f in features]

# Confidence / selectivity metrics for pooled (overall) classification.
obs = ov[obs_cols].to_numpy(dtype=float)
thr = ov[thr_cols].to_numpy(dtype=float)
pvals = ov[p_cols].to_numpy(dtype=float)

obs_filled = np.where(np.isfinite(obs), obs, -np.inf)
pref_idx = np.argmax(obs_filled, axis=1)
pref_feature = np.array(features, dtype=object)[pref_idx]
pref_obs = obs[np.arange(len(ov)), pref_idx]
pref_thr = thr[np.arange(len(ov)), pref_idx]
pref_p = pvals[np.arange(len(ov)), pref_idx]

runner = obs_filled.copy()
runner[np.arange(len(ov)), pref_idx] = -np.inf
runner_obs = np.max(runner, axis=1)
runner_obs = np.where(np.isfinite(runner_obs), runner_obs, np.nan)

selectivity_index = (pref_obs - runner_obs) / (np.abs(pref_obs) + np.abs(runner_obs) + 1e-12)
shuffle_confidence = 1.0 - pref_p
margin_to_thr99 = pref_obs - pref_thr
norm_margin_to_thr99 = margin_to_thr99 / (np.abs(pref_thr) + 1e-12)

ov["preferred_feature"] = pref_feature
ov["pref_obs"] = pref_obs
ov["pref_thr99"] = pref_thr
ov["pref_p"] = pref_p
ov["selectivity_index"] = selectivity_index
ov["shuffle_confidence"] = shuffle_confidence
ov["margin_to_thr99"] = margin_to_thr99
ov["norm_margin_to_thr99"] = norm_margin_to_thr99
ov["certainty_score"] = ov["selectivity_index"] * ov["shuffle_confidence"]
ov["is_significant_pref"] = ov["pref_p"] < 0.01
ov["overall_encoding"] = np.where(ov["is_significant_pref"], ov["preferred_feature"], "none")

# Attach neuron location from SpikeClusterMetadata (mode across sessions per unit).
mdf = meta_data["SpikeClusterMetadata"].reset_index().copy()

area_candidates = ["fine_brain_area", "brain_area", "region", "area"]
area_col = next((c for c in area_candidates if c in mdf.columns), None)
if area_col is None:
    raise KeyError(
        "Could not find region column in SpikeClusterMetadata. "
        f"Tried: {area_candidates}"
    )

mdf["brain_region"] = (
    mdf[area_col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan}).fillna("Unknown")
)


def _normalize_unit_from_series(series: pd.Series) -> pd.Series:
    txt = series.astype(str).str.strip()
    from_label = txt.str.extract(r"(?i)unit\s*0*(\d+)", expand=False)
    from_label_num = pd.to_numeric(from_label, errors="coerce")
    from_num = pd.to_numeric(series, errors="coerce")
    unit_num = from_label_num.combine_first(from_num)
    return unit_num.map(lambda v: f"Unit{int(v):04d}" if pd.notna(v) else np.nan)


unit_source_col = None
for cand in ["unit", "unit_id", "unit_name", "cluster_id", "entry_id"]:
    if cand in mdf.columns:
        u = _normalize_unit_from_series(mdf[cand])
        if u.notna().any():
            mdf["unit"] = u
            unit_source_col = cand
            break

if "unit" not in mdf.columns or mdf["unit"].notna().sum() == 0:
    raise KeyError(
        "Could not construct unit labels from SpikeClusterMetadata. "
        "Expected one of unit/unit_id/unit_name/cluster_id/entry_id."
    )

if unit_source_col == "entry_id":
    shifted = _normalize_unit_from_series(mdf["entry_id"] + 1)
    match_default = mdf["unit"].isin(ov["unit"]).sum()
    match_shifted = shifted.isin(ov["unit"]).sum()
    if match_shifted > match_default:
        mdf["unit"] = shifted

unit_region = (
    mdf[["unit", "brain_region"]]
    .dropna(subset=["unit"])
    .groupby("unit", as_index=False)["brain_region"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0])
)

ov = ov.merge(unit_region, on="unit", how="left")
ov["brain_region"] = ov["brain_region"].fillna("Unknown")

enc_order = ["cue", "reward_approach", "reward", "none"]
ov["enc_rank"] = pd.Categorical(ov["overall_encoding"], categories=enc_order, ordered=True)
ov = ov.sort_values(["enc_rank", "brain_region", "certainty_score"], ascending=[True, True, False]).reset_index(drop=True)
ov["unit_region"] = ov["unit"] + " | " + ov["brain_region"]

# Strength relative to significance threshold for each feature (positive means above 99th shuffle threshold).
for f in features:
    ov[f"strength_{f}"] = (ov[f"obs_{f}"] - ov[f"thr99_{f}"]) / (np.abs(ov[f"thr99_{f}"]) + 1e-12)

# Plot 1: Which neuron accounts for what (overall), with location in the label.
fig_assignment = px.strip(
    ov,
    x="overall_encoding",
    y="unit_region",
    color="overall_encoding",
    orientation="h",
    category_orders={"overall_encoding": enc_order},
    hover_data={
        "unit": True,
        "brain_region": True,
        "preferred_feature": True,
        "pref_p": ":.4g",
        "certainty_score": ":.4f",
        "selectivity_index": ":.4f",
        "shuffle_confidence": ":.4f",
    },
    title="Overall class assignment per neuron (labels include brain region)",
    labels={
        "overall_encoding": "Overall encoding class",
        "unit_region": "Neuron | Region",
    },
)
fig_assignment.update_layout(template="plotly_white", height=max(550, 13 * len(ov)))
fig_assignment.show()

# Plot 2: Location summary by overall encoding class.
region_order = ov["brain_region"].value_counts().index.tolist()
region_counts = (
    ov.groupby(["brain_region", "overall_encoding"], as_index=False)
    .size()
    .rename(columns={"size": "n_neurons"})
)

fig_region = px.bar(
    region_counts,
    x="brain_region",
    y="n_neurons",
    color="overall_encoding",
    barmode="stack",
    category_orders={"overall_encoding": enc_order, "brain_region": region_order},
    title="Neuron locations: counts per brain region and overall encoding",
    labels={
        "brain_region": "Brain region",
        "n_neurons": "Number of neurons",
        "overall_encoding": "Overall encoding",
    },
)
fig_region.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_region.show()

# Plot 3: Confidence of overall classification.
ov["confidence_size"] = np.clip(np.nan_to_num(ov["norm_margin_to_thr99"], nan=0.0), 0.0, 3.0) + 0.2
fig_conf = px.scatter(
    ov,
    x="selectivity_index",
    y="shuffle_confidence",
    color="overall_encoding",
    size="confidence_size",
    hover_data={
        "unit": True,
        "brain_region": True,
        "preferred_feature": True,
        "pref_p": ":.4g",
        "margin_to_thr99": ":.4f",
        "certainty_score": ":.4f",
        "is_significant_pref": True,
    },
    category_orders={"overall_encoding": enc_order},
    title="Overall classification confidence per neuron",
    labels={
        "selectivity_index": "Selectivity vs next-best feature",
        "shuffle_confidence": "Shuffle confidence (1 - p)",
        "overall_encoding": "Overall encoding",
        "confidence_size": "Norm. margin over thr99",
    },
)
fig_conf.update_layout(template="plotly_white")
fig_conf.update_yaxes(range=[0, 1])
fig_conf.add_hline(y=0.99, line_dash="dash", line_color="gray")
fig_conf.show()

overall_classification_with_region = ov.copy()

display(
    overall_classification_with_region[
        [
            "unit",
            "brain_region",
            "overall_encoding",
            "preferred_feature",
            "pref_p",
            "shuffle_confidence",
            "certainty_score",
            "assigned_label",
        ]
    ].head(20)
)



/var/folders/ml/fyq5x5t55wx4129dn03n32hm0000gn/T/ipykernel_43827/372626467.py:79: FutureWarning:

The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.



,unit,brain_region,overall_encoding,preferred_feature,pref_p,shuffle_confidence,certainty_score,assigned_label
0,Unit0020,DG,cue,cue,0.008991,0.991009,0.056882,cue
1,Unit0011,DG,cue,cue,0.004995,0.995005,0.054629,cue
2,Unit0015,DG,cue,cue,0.000999,0.999001,0.027636,cue+reward_approach
3,Unit0061,InfrL,cue,cue,0.000999,0.999001,0.164698,cue
4,Unit0063,InfrL,cue,cue,0.000999,0.999001,0.121586,cue
5,Unit0064,InfrL,cue,cue,0.000999,0.999001,0.045890,cue
6,Unit0072,InfrL,cue,cue,0.001998,0.998002,0.028621,cue
7,Unit0066,InfrL,cue,cue,0.000999,0.999001,0.015748,cue
8,Unit0040,PrL,cue,cue,0.001998,0.998002,0.152953,cue
9,Unit0056,PrL,cue,cue,0.000999,0.999001,0.120572,cue
